In [1]:
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [2]:
def plot_distribution(df, attribute, coloring, run, relation, combination, acc):
    palette ={"nlr": "C0", "busco": "C1", "housekeeping": "C2", "all": "C3", "rest":"C3"}
    order=["nlr","busco","housekeeping"] if combination == "reference" else ["nlr","busco","housekeeping", "all"] if combination == "background" else ["nlr","busco","housekeeping", "rest"]
    sns.displot(df, x=attribute, hue = coloring, kind="kde",common_norm=False, palette=palette, hue_order=order)
    plt.savefig("{}/{}/{}/{}_pb_augustus_{}_{}.png".format(run,acc,relation,acc,combination,attribute), bbox_inches='tight')
    plt.close()

In [3]:
def plot_categories(df, attribute, coloring, run, relation, combination, acc):
    palette ={"nlr": "C0", "busco": "C1", "housekeeping": "C2", "all": "C3", "rest":"C3"}
    order=["nlr","busco","housekeeping"] if combination == "reference" else ["nlr","busco","housekeeping", "all"] if combination == "background" else ["nlr","busco","housekeeping", "rest"]
    sns.histplot(df, x=attribute, hue =coloring, multiple= "dodge", stat="percent", common_norm=False, shrink = 0.8,  palette=palette, hue_order=order)
    plt.xticks(rotation=45, horizontalalignment='right')
    plt.savefig("{}/{}/{}/{}_pb_augustus_{}_{}.png".format(run,acc,relation,acc,combination,attribute), bbox_inches='tight')
    plt.close()

# Preparation of reference gene lists

In [4]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900','Araport']
paths=["/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/busco_Araport11_20221221_augustus_unique.txt",
       "/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/arat_housekeeping_augustus_unique.txt",
       "/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/nlrs_in_orthogroups_20221221_augustus_unique.txt"]
ref=["busco","housekeeping","nlrs"]

for path, ref in zip(paths, ref):
    file = open(path, 'r')
    Lines = file.readlines()

    mappings = {}
    for acc in accessions:
        mappings[acc]=[]

    # Strips the newline character
    for line in Lines:
        a2g = line.strip().split(": ")[1].split(" ")
        for c in a2g:
            # print(c.split("_")[1])
            acc = c.split("_")[0]
            gene = c.split("_")[1]
            try:
                mappings[acc].append(gene)
            except:
                continue
                
    for acc in mappings.keys():
        if acc != "Araport":
            with open(f'ref_genes/augustus_mappings/{ref}_{acc}.txt', 'a') as out:
                for gene in mappings[acc]:
                    out.write(f'{gene}\n')
        else:
            with open(f'ref_genes/augustus_mappings/{ref}_col0.txt', 'a') as out:
                for gene in mappings[acc]:
                    out.write(f'{gene}\n')

# FULL RUN

In [18]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900','col0']
#accessions= ['at6923']

#runs=["full","iso2","gene2"]
#runs=["full"]
runs=["gene2"]

for run in tqdm(runs):
    pb_augustus_df = pd.DataFrame()
    pb_subcat = pd.DataFrame()
    for acc in tqdm(accessions):
        # -------------0.5-----------
        # prepare ref gene sets:
        busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/busco_{acc}.txt", header=None)
        house = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/housekeeping_{acc}.txt", header=None)
        nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/nlrs_{acc}.txt", header=None)
        
        busco["type"] = "busco"
        house["type"] = "housekeeping"
        nlr["type"] = "nlr"
        gene_type = pd.concat([busco,house,nlr])
        gene_type.columns = ["ref_isoform","type"]
        gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

        gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
        gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"
        
        # -------------0-------------
        # make directories
        os.makedirs("{}/{}/isoform".format(run,acc), exist_ok=True)
        os.makedirs("{}/{}/gene".format(run,acc), exist_ok=True)
        # -------------1-------------
        # prepare sqanti output
        sqanti = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_pb/{}_augustus_sqanti/{}-pb-vs-augustus_classification.txt".format(acc,acc), sep="\t")
        sub = sqanti[["isoform","exons","length","structural_category","associated_gene","associated_transcript","subcategory","ORF_seq"]]
        sub["gene"] = sub["isoform"].str.split(".").str[0]+"."+sub["isoform"].str.split(".").str[1]
        # -------------3-------------
        # prepare read support
        reads=pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_pb/{acc}.collapsed.pb.v2.abundance.txt",sep="\t", skiprows=3)
        r_sub = reads[["pbid","fl_assoc"]]
        r_sub.columns = ["isoform", "isoform_reads"]
        r_sub["gene"]=r_sub["isoform"].str.split(".").str[0]+"."+r_sub["isoform"].str.split(".").str[1]
        for i in r_sub.groupby("gene"):
            r_sub.loc[r_sub["gene"]==i[0], "gene_reads"] = i[1]["isoform_reads"].sum()
        r_sub["gene_reads"] = r_sub["gene_reads"].astype("int")

        # -------------4-------------
        # add accession ID and merge all information
        merged_all = pd.merge(left=sub, right=r_sub[["isoform","isoform_reads","gene_reads"]], how='left', left_on='isoform', right_on='isoform')
        merged_all['accession'] = acc

        # set dataset to work with
        if run == "iso2":
            merged_all=merged_all[merged_all["isoform_reads"]>1]
        elif run == "gene2":
            merged_all=merged_all[merged_all["gene_reads"]>1]

        pb_augustus_df = pd.concat([pb_augustus_df, merged_all])

        # -------------5-------------
        # relate the gene type to their isoform in sqanti
        iso_ref = pd.merge(left=merged_all, right=gene_type, how='left', left_on='associated_transcript', right_on='ref_isoform')
        # ------------5.1------------
        # basic analysis
        iso_nlr = iso_ref.loc[iso_ref["type"] == "nlr"]
        iso_busco = iso_ref.loc[iso_ref["type"] == "busco"]
        iso_house = iso_ref.loc[iso_ref["type"] == "housekeeping"]
        iso_all = pd.concat([iso_nlr,iso_busco,iso_house])
        # plot exon + length distribution
        plot_distribution(iso_all, "exons", "type", run, "isoform", "reference", acc)
        plot_distribution(iso_all, "length", "type", run,  "isoform", "reference", acc)
        # plot category and subcategory
        plot_categories(iso_all, "structural_category", "type", run, "isoform", "reference", acc)
        plot_categories(iso_all, "subcategory", "type", run,  "isoform", "reference", acc)
        # ------------5.2------------
        # all genes as background, reference genes on top
        gene_background = merged_all.copy()
        gene_background["ref_isoform"] = "None"
        gene_background["type"] = "all"
        gene_background["ref_gene"] = "None"
        iso_background_all = pd.concat([iso_all,gene_background], ignore_index=True)
        # plot exon + length distribution
        plot_distribution(iso_background_all, "exons", "type", run, "isoform", "background", acc)
        plot_distribution(iso_background_all, "length", "type", run,  "isoform", "background", acc)
        # plot category and subcategory
        plot_categories(iso_background_all, "structural_category", "type", run, "isoform", "background", acc)
        plot_categories(iso_background_all, "subcategory", "type", run,  "isoform", "background", acc)
        # ------------5.3------------
        # rest vs. busco vs. house vs nlr.
        iso_ava = iso_ref.fillna("rest")
        # plot exon + length distribution
        plot_distribution(iso_ava, "exons", "type", run, "isoform", "ava", acc)
        plot_distribution(iso_ava, "length", "type", run,  "isoform", "ava", acc)
        # plot category and subcategory
        plot_categories(iso_ava, "structural_category", "type", run, "isoform", "ava", acc)
        plot_categories(iso_ava, "subcategory", "type", run,  "isoform", "ava", acc)

        # -------------6-------------
        # relate the gene type to their gene in sqanti
        gene_ref = pd.merge(left=merged_all, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
        # ------------6.1------------
        # basic analysis
        gene_nlr = gene_ref.loc[gene_ref["type"] == "nlr"]
        gene_busco = gene_ref.loc[gene_ref["type"] == "busco"]
        gene_house = gene_ref.loc[gene_ref["type"] == "housekeeping"]
        gene_all = pd.concat([gene_nlr,gene_busco,gene_house])
        # plot exon + length distribution
        plot_distribution(gene_all, "exons", "type", run, "gene", "reference", acc)
        plot_distribution(gene_all, "length", "type", run,  "gene", "reference", acc)
        # plot category and subcategory
        plot_categories(gene_all, "structural_category", "type", run, "gene", "reference", acc)
        plot_categories(gene_all, "subcategory", "type", run,  "gene", "reference", acc)
        # ------------6.2------------
        # all genes as background, reference genes on top
        gene_background = merged_all.copy()
        gene_background["ref_isoform"] = "None"
        gene_background["type"] = "all"
        gene_background["ref_gene"] = "None"
        gene_background_all = pd.concat([gene_all,gene_background], ignore_index=True)
        # plot exon + length distribution
        plot_distribution(gene_background_all, "exons", "type", run, "gene", "background", acc)
        plot_distribution(gene_background_all, "length", "type", run,  "gene", "background", acc)
        # plot category and subcategory
        plot_categories(gene_background_all, "structural_category", "type", run, "gene", "background", acc)
        plot_categories(gene_background_all, "subcategory", "type", run,  "gene", "background", acc)
        # ------------6.3------------
        # rest vs. busco vs. house vs nlr.
        gene_ava = gene_ref.fillna("rest")
        # plot exon + length distribution
        plot_distribution(gene_ava, "exons", "type", run, "gene", "ava", acc)
        plot_distribution(gene_ava, "length", "type", run,  "gene", "ava", acc)
        # plot category and subcategory
        plot_categories(gene_ava, "structural_category", "type", run, "gene", "ava", acc)
        plot_categories(gene_ava, "subcategory", "type", run,  "gene", "ava", acc)

        # -------------7-------------
        # prepare the subcategory dataframe:
        idx=sub["structural_category"].value_counts().index
        cat_df = pd.DataFrame({"accession":[acc]})
        for cat in idx:
            cat_df[cat]=sub["structural_category"].value_counts()[cat]
        test=cat_df.set_index("accession").T[acc].sum()
        cat_df["sum"] = cat_df.set_index("accession").T[acc].sum()
        pb_subcat=pd.concat([pb_subcat, cat_df], ignore_index=True).fillna(0)
        
    pb_augustus_df = pb_augustus_df[["gene","isoform","accession","associated_gene","associated_transcript","exons","length","isoform_reads","gene_reads","structural_category","subcategory","ORF_seq"]]
    pb_augustus_df.columns = ["gene","isoform","accession","associated_gene","associated_transcript","exons","length","isoform_read_support","gene_read_support","structural_category","subcategory","Sqanti_ORF_seq"]
    if run == "full":
        pb_augustus_df.to_pickle("./pb_augustus_sqanti_{}.pkl".format(run))
        pb_augustus_df.to_csv("./pb_augustus_sqanti_{}.tsv".format(run), sep="\t")

    # run analysis over the full dataframes
    os.makedirs("{}/{}/gene".format(run,"all"), exist_ok=True)
    
    # basic
    all_gene_ref = pd.merge(left=pb_augustus_df, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
    all_gene_nlr = all_gene_ref.loc[all_gene_ref["type"] == "nlr"]
    all_gene_busco = all_gene_ref.loc[all_gene_ref["type"] == "busco"]
    all_gene_house = all_gene_ref.loc[all_gene_ref["type"] == "housekeeping"]
    all_gene_all = pd.concat([all_gene_nlr,all_gene_busco,all_gene_house])
    # plot exon + length distribution
    plot_distribution(all_gene_all, "exons", "type", run, "gene", "reference", "all")
    plot_distribution(all_gene_all, "length", "type", run, "gene", "reference", "all")
    # plot category and subcategory
    plot_categories(all_gene_all, "structural_category", "type", run, "gene", "reference", "all")
    plot_categories(all_gene_all, "subcategory", "type", run,  "gene", "reference", "all")

    # background
    all_background = pb_augustus_df.copy()
    all_background["ref_isoform"] = "None"
    all_background["type"] = "all"
    all_background["ref_gene"] = "None"
    all_background_all = pd.concat([all_gene_all,all_background], ignore_index=True)
    # plot exon + length distribution
    plot_distribution(all_background_all, "exons", "type", run, "gene", "background", "all")
    plot_distribution(all_background_all, "length", "type", run, "gene", "background", "all")
    # plot category and subcategory
    plot_categories(all_background_all, "structural_category", "type", run, "gene", "background", "all")
    plot_categories(all_background_all, "subcategory", "type", run,  "gene", "background", "all")

    # all vs all
    all_gene_ava = all_gene_ref.fillna("rest")
    # plot exon + length distribution
    plot_distribution(all_gene_ava, "exons", "type", run, "gene", "ava", "all")
    plot_distribution(all_gene_ava, "length", "type", run, "gene", "ava", "all")
    # plot category and subcategory
    plot_categories(all_gene_ava, "structural_category", "type", run, "gene", "ava", "all")
    plot_categories(all_gene_ava, "subcategory", "type", run,  "gene", "ava", "all")
    
    #t = pb_subcat.copy()
    for cat in pb_subcat.set_index("accession").columns:
        pb_subcat["proportion"]=pb_subcat[cat]/pb_subcat["sum"]*100
        sns.barplot(data=pb_subcat, x="accession", y="proportion")
        plt.xticks(rotation=45, horizontalalignment='right')
        plt.ylabel("{} [%]".format(cat))
        plt.savefig("{}/all/all_pb_augustus_{}.png".format(run,cat), bbox_inches='tight')
        plt.close()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [1:48:17<00:00, 6497.42s/it]


In [8]:
test=pd.read_pickle("pb_augustus_sqanti_full.pkl")

In [ ]:
sns.displot(test, x="exons", hue = "accession", kind="kde",common_norm=False)
plt.savefig("pb_augustus_exons_full.png", bbox_inches='tight')
plt.close()

In [ ]:
sns.displot(test, x="length", hue = "accession", kind="kde",common_norm=False)
plt.savefig("pb_augustus_length_full.png", bbox_inches='tight')
plt.close()